<a href="https://colab.research.google.com/github/ZehanQin/ECON5200-Applied-Data-Analytics-in-Econ/blob/main/Lab%2025/lab_ch25_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 25: AI Literacy & Generative Tools — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 45 min Portfolio + 45 min RAG Pipeline

---

**Format:** Part 1 builds a portfolio site with Claude Code (faster-paced than 3916). Part 2 presents a **deliberately broken RAG pipeline** that you must diagnose, fix, and extend.

**Learning Objectives:**
- Deploy a professional portfolio website using AI-assisted development
- Diagnose configuration errors in a Retrieval-Augmented Generation pipeline
- Compare RAG-grounded answers to direct LLM answers on economic questions
- Evaluate retrieval quality with precision and relevance metrics

**Tools:** Claude Code CLI, Next.js, Vercel, LangChain, ChromaDB, OpenAI API

**Time estimate:** ~90 minutes total

---

---

## Part 1: Portfolio Site (45 min)

Build and deploy a professional portfolio site with Claude Code. You are expected to move quickly — the setup and scaffold steps are compressed.

### Setup (5 min)

```bash
# Install Claude Code CLI (if not already installed)
npm install -g @anthropic-ai/claude-code
claude login
claude skill install /build-portfolio-site
```

Create a [Vercel account](https://vercel.com) via GitHub SSO if you do not have one.

### Scaffold & Customize (25 min)

```bash
mkdir econ-portfolio && cd econ-portfolio
claude
```

In the Claude Code session, provide your resume content, style preferences (browse [stitch.withgoogle.com](https://stitch.withgoogle.com) for reference), and at least **3 projects** from your coursework. Your prompt should include:

- Name, program, professional bio
- Style reference (colors, layout preference)
- Project titles, descriptions, GitHub links, and technologies
- A custom section (choose: Data Projects gallery, Research Interests, or Skills display)

### Deploy (10 min)

```bash
git init && git add . && git commit -m "Initial portfolio"
gh repo create econ-portfolio --public --push
```

Import to Vercel at [vercel.com/new](https://vercel.com/new). Test on desktop and mobile.

### Reflection (5 min)

In a markdown cell below, briefly answer:
1. What prompts worked best? What required iteration?
2. What did you have to fix that the AI got wrong?
3. How does this compare to writing the site from scratch?

**Your live URL:** `https://econ-portfolio-delta.vercel.app/


*Part 1 Reflection:*

Vercel URL:
https://econ-portfolio-delta.vercel.app/

Github Repo URL:
https://github.com/ZehanQin/econ-portfolio/tree/main/app

1. The most effective prompt is to send my resume, transcript, style of the website, target model and technology in one setting and ensure its clear. Since in this case AI would gain a perfect understanding about who am I, what kind of website is my target. Therefore, the first version of the website structure is clear and complete. What required iteration is the detail configuration and website effects. For instance, the intialization and whether is the style fitted my expectations. This leading to keep feeding prompts to the AI and continuous adjustment.
2. In the beginning, AI wrote the configuration document for Next.js into next.config.ts, while Next.js 14 does not allowed such format, which leading to a error during the compilation. Eventually I have to fix accordingly to the error information and turning it into a correct .mjs document.
3. The efficiency is on another level, comparing to writing it by myself, AI can structure the framework, website style and basic content directly instead of me starting from zero. However, as what the class mentioned, we are rather a supervisor instead of builder, it is important for us to check error and keep adjusting the details to fit the target of our expectation.

---

## Part 2: RAG Pipeline — Diagnose, Fix, Extend (45 min)

The code below implements a Retrieval-Augmented Generation (RAG) pipeline that ingests FOMC minutes (from your Ch 23 lab) and answers economic questions grounded in those documents.

**The pipeline has 3 deliberate errors:**
1. A **chunking configuration** error
2. An **embedding model** error
3. A **retrieval parameter** error

Your job: run the code, identify each error, explain why it matters, and fix it.

In [1]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Install required packages
# -----------------------------------------------------------
!pip install langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface sentence-transformers chromadb tiktoken -q

import os
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

print('Libraries loaded. Ready to diagnose.')

Libraries loaded. Ready to diagnose.


Note: Since pre-lab preparation point out pip install langchain langchain-community chromadb sentence-transformers, so no need for Open_Ai_Key thus changed original code.

In [2]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Load 5 sample FOMC minutes (simulated for lab purposes)
# In your Ch 23 lab you scraped real FOMC minutes; here we
# use representative excerpts for reproducibility.
# -----------------------------------------------------------

fomc_documents = [
    Document(
        page_content="""Federal Open Market Committee - January 2023 Meeting.
        Participants noted that inflation remained well above the Committee's
        2 percent objective. The labor market remained very tight, with the
        unemployment rate near historical lows. Wage growth had remained
        elevated relative to what would be consistent with 2 percent inflation
        over time given prevailing trends in productivity growth. Participants
        agreed that the Committee should continue to raise the target range
        for the federal funds rate at upcoming meetings. Several participants
        noted the risk that the lagged effects of cumulative policy tightening
        could end up being more restrictive than necessary.""",
        metadata={"meeting": "January 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - March 2023 Meeting.
        Recent banking sector developments were likely to result in tighter
        credit conditions for households and businesses. Participants
        discussed the effects of the recent banking stress on the economic
        outlook, noting that tighter credit conditions would likely weigh on
        economic activity. Some participants noted that monetary policy
        actions and banking stress were working in the same direction to slow
        the economy. Inflation remained elevated, and recent data provided
        few signs that inflationary pressures were abating quickly enough.""",
        metadata={"meeting": "March 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - June 2023 Meeting.
        Participants noted that the pace of job gains had slowed but remained
        solid. Consumer spending appeared to have picked up. The housing sector
        remained weak, reflecting higher mortgage rates. Participants expected
        inflation to come down as the effects of tight monetary policy worked
        through the economy, though the process was expected to take time.
        Participants generally judged that, with inflation still well above
        the Committee's longer-run goal of 2 percent, keeping the federal
        funds rate at a restrictive level was appropriate.""",
        metadata={"meeting": "June 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - September 2023 Meeting.
        Participants discussed the uncertainty surrounding the economic outlook
        and noted various risks. Supply-side improvements had contributed to
        the decline in inflation. Labor market conditions had eased somewhat
        but remained tight. Several participants emphasized the importance of
        communicating that the Committee would proceed carefully in determining
        the extent of additional policy firming. The Committee decided to
        maintain the target range for the federal funds rate at 5-1/4 to
        5-1/2 percent.""",
        metadata={"meeting": "September 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - December 2023 Meeting.
        Participants noted that the disinflation process was continuing.
        The labor market remained strong with solid job gains and the
        unemployment rate remaining low. Several participants pointed to
        the risk that progress on inflation could stall. The median
        projection for the federal funds rate at end of 2024 was 4.6 percent,
        suggesting rate cuts could begin in 2024. Participants emphasized
        that the timing and pace of rate cuts would depend on incoming data
        and the evolving economic outlook.""",
        metadata={"meeting": "December 2023", "year": 2023}
    ),
]

print(f'Loaded {len(fomc_documents)} FOMC minutes documents.')
for doc in fomc_documents:
    print(f"  - {doc.metadata['meeting']}: {len(doc.page_content)} characters")

Loaded 5 FOMC minutes documents.
  - January 2023: 731 characters
  - March 2023: 649 characters
  - June 2023: 649 characters
  - September 2023: 617 characters
  - December 2023: 609 characters


### DIAGNOSE: Error 1 — Chunking Configuration

The text splitter below has **two problems** with its configuration. Run the cell and examine the output to identify them.

In [3]:
# -----------------------------------------------------------
# DIAGNOSE: This code has errors in the chunking configuration.
# -----------------------------------------------------------

# ERROR 1a: chunk_size is far too small — 50 characters means each
# chunk is just a few words, destroying semantic coherence.
# A good chunk_size for RAG is typically 500-1000 characters.
#
# ERROR 1b: chunk_overlap is 0 — sentences at chunk boundaries
# get split in the middle, losing context. A typical overlap
# is 10-20% of chunk_size (e.g., 100 for chunk_size=500).

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,        # BUG: way too small!
    chunk_overlap=0,      # BUG: no overlap means lost context at boundaries
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(fomc_documents)

print(f'Number of chunks: {len(chunks)}')
print(f'Average chunk length: {np.mean([len(c.page_content) for c in chunks]):.0f} characters')
print(f'\nFirst 5 chunks:')
for i, chunk in enumerate(chunks[:5]):
    print(f'  Chunk {i}: "{chunk.page_content[:80]}..." ({len(chunk.page_content)} chars)')

print(f'\nPROBLEM: With 50-character chunks, each chunk is just a sentence fragment.')
print('The retriever will match on fragments, not meaningful passages.')

Number of chunks: 97
Average chunk length: 29 characters

First 5 chunks:
  Chunk 0: "Federal Open Market Committee - January 2023..." (44 chars)
  Chunk 1: "Meeting...." (8 chars)
  Chunk 2: "Participants noted that inflation..." (33 chars)
  Chunk 3: "remained well above the Committee's..." (35 chars)
  Chunk 4: "2 percent objective..." (19 chars)

PROBLEM: With 50-character chunks, each chunk is just a sentence fragment.
The retriever will match on fragments, not meaningful passages.


Diagnosis: chunk_size=50 and chunk_overlap=0 are wrong. It affected RAG quality since 50 equivalent to around 9 words, which shorter than a sentence. In reality it runs out 97 chunks, which averagely around 29 characters. Informations like "Meeting...." are not count as semantic unit. Therefore, it can only do vector matching though such fragments, which lead to the top-k's result of high similarity score while small amount of informations. On the other hand, chunk_overlap=0 means that sentences that are cross boundaries would be cut off, thus subject and predicate would be thrown into different vectors. We should fix it to chunk_size=500, chunk_overlap=100.

### DIAGNOSE: Error 2 — Embedding Model Mismatch

The embedding model below is wrong for this task. Identify the issue.

In [4]:
# -----------------------------------------------------------
# DIAGNOSE: The embedding model is wrong.
# -----------------------------------------------------------

# ERROR 2: Using paraphrase-MiniLM-L3-v2 which is a small, outdated
# retrieval model (3 transformer layers, 384 dims). It scores poorly
# on the MTEB retrieval benchmark compared to all-mpnet-base-v2.
# (This mirrors the original ada-002 vs text-embedding-3-small bug.)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L3-v2"  # BUG: weak retrieval model
)

# Create vector store from (badly chunked) documents
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="fomc_minutes_broken"
)

print(f'Vector store created with {vectorstore._collection.count()} vectors.')
print(f'Embedding model: paraphrase-MiniLM-L3-v2 (weak retrieval model)')
print(f'Embedding dimensions: 384')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created with 97 vectors.
Embedding model: paraphrase-MiniLM-L3-v2 (weak retrieval model)
Embedding dimensions: 384


Diagnosis: The embedding model chosed paraphrase-MiniLM-L3-v2, which is old and small, since only 3 transformer, 384 embedding, it is not for retrieval optimization. The reason that it affects RAG quality is due to embedding model determines whether texts with similar meanings are truly close within the vector space. The weak model would create high similarity score noise, thus retrieve precision would decreased. For instance, the chunk might get lined heind unrelated chunk. To fix it, we should change to all-mpnet-base-v2, which 12 transformer, 768 dimensions, and it would score significantly higher on the MTEB retrieval benchmark.

Note: due to the pre-lab preparation requirement of sentence-transformers, chose wrong and right model according to the HuggingFace local model however, the bug diagosis is in a similar logic.

### DIAGNOSE: Error 3 — Retrieval Parameter

The retriever configuration has a parameter that makes retrieval ineffective.

In [5]:
# -----------------------------------------------------------
# DIAGNOSE: The retrieval parameter is wrong.
# -----------------------------------------------------------

# ERROR 3: k=1 means we only retrieve ONE chunk. For a question
# that spans multiple meetings or topics, a single chunk is
# almost never sufficient. Typical k values are 3-5 for
# small corpora and 5-10 for larger ones.

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}  # BUG: only retrieves 1 chunk!
)

# Build a simple RAG answerer (no LLM API — uses retrieved context directly)
def rag_answer(query, retriever):
    docs = retriever.invoke(query)
    meetings = [d.metadata.get("meeting", "unknown") for d in docs]
    answer = f"Based on {len(docs)} retrieved FOMC minute(s) ({', '.join(meetings)}), the relevant context is:\n\n"
    for d in docs:
        answer += f"- [{d.metadata.get('meeting','unknown')}] {d.page_content.strip()[:200]}...\n"
    return {"result": answer, "source_documents": docs}

# Test with a question that requires multiple meetings
test_query = "How did the Fed's stance on inflation evolve throughout 2023?"
result = rag_answer(test_query, retriever)

print(f'Question: {test_query}')
print(f'\nAnswer: {result["result"]}')
print(f'\nSources retrieved: {len(result["source_documents"])}')
for i, doc in enumerate(result["source_documents"]):
    print(f'  Source {i+1}: {doc.metadata.get("meeting", "unknown")} — '
          f'"{doc.page_content[:60]}..."')
print(f'\nPROBLEM: Only 1 source retrieved. This question requires context')
print('from multiple meetings to trace the evolution of Fed policy.')

Question: How did the Fed's stance on inflation evolve throughout 2023?

Answer: Based on 1 retrieved FOMC minute(s) (June 2023), the relevant context is:

- [June 2023] inflation to come down as the effects of...


Sources retrieved: 1
  Source 1: June 2023 — "inflation to come down as the effects of..."

PROBLEM: Only 1 source retrieved. This question requires context
from multiple meetings to trace the evolution of Fed policy.


Diagnosis: The bug happened since k=1 make retriever only return one chunk. It affected RAG quality since test query type of questions need multiple meetings' information to answer due to the question structure. According to the result, only June 2023 output is shown, Janunary, March, September and so on are missed. Therefore, LLM could not trace timeline, while RAG's main value is to place answer to the according retrieved source. However, k=1 made RAG uneffective. To fix it, k=4, k=3-5 for small data, this would cover multiple meetings as well as maintained the clean prompt.

---

## YOUR TASK: Fix the RAG Pipeline

Correct all three errors and rebuild the pipeline from scratch.

**Verification checkpoints:**
- Chunks should average 400-600 characters with ~100 character overlap
- Embedding model should be `text-embedding-3-small`
- Retriever should return k=4 or k=5 documents
- Test query about Fed stance evolution should cite 3+ meetings

In [6]:
# -----------------------------------------------------------
# YOUR TASK — Fix all three errors and rebuild the pipeline
# -----------------------------------------------------------

# Fix 1: Correct chunking parameters
text_splitter_fixed = RecursiveCharacterTextSplitter(
    chunk_size=500,          # FILL IN: appropriate chunk size
    chunk_overlap=100,       # FILL IN: appropriate overlap
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_fixed = text_splitter_fixed.split_documents(fomc_documents)
print(f'Fixed chunks: {len(chunks_fixed)}')
print(f'Avg chunk length: {np.mean([len(c.page_content) for c in chunks_fixed]):.0f} chars')


# Fix 2: Correct embedding model
embeddings_fixed = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"   # FILL IN: current recommended model
)

vectorstore_fixed = Chroma.from_documents(
    documents=chunks_fixed,
    embedding=embeddings_fixed,
    collection_name="fomc_minutes_fixed"
)


# Fix 3: Correct retrieval parameter
retriever_fixed = vectorstore_fixed.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # FILL IN: appropriate k value
)

# Build the QA chain (uses our rag_answer function since we don't have an LLM API)

# Verification: re-run the same test query
result_fixed = rag_answer(test_query, retriever_fixed)
print(f'\nFixed Answer: {result_fixed["result"]}')
print(f'Sources retrieved: {len(result_fixed["source_documents"])}')
for i, doc in enumerate(result_fixed["source_documents"]):
    print(f'  Source {i+1}: {doc.metadata.get("meeting", "unknown")}')

Fixed chunks: 10
Avg chunk length: 360 chars


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fixed Answer: Based on 4 retrieved FOMC minute(s) (January 2023, September 2023, June 2023, December 2023), the relevant context is:

- [January 2023] Federal Open Market Committee - January 2023 Meeting.
        Participants noted that inflation remained well above the Committee's
        2 percent objective. The labor market remained very tight, w...
- [September 2023] Federal Open Market Committee - September 2023 Meeting.
        Participants discussed the uncertainty surrounding the economic outlook
        and noted various risks. Supply-side improvements had co...
- [June 2023] Federal Open Market Committee - June 2023 Meeting.
        Participants noted that the pace of job gains had slowed but remained
        solid. Consumer spending appeared to have picked up. The housin...
- [December 2023] Federal Open Market Committee - December 2023 Meeting.
        Participants noted that the disinflation process was continuing.
        The labor market remained strong with solid job g

---

## EXTEND: Query the Pipeline with 3 Economic Questions

Ask your fixed pipeline 3 questions about the 2023 FOMC minutes. For each question, evaluate:
1. **Retrieval quality** — Did the retriever find the right source documents?
2. **Answer grounding** — Is the answer supported by the retrieved text, or does it hallucinate?
3. **Completeness** — Does the answer address all parts of the question?

In [7]:
# -----------------------------------------------------------
# YOUR TASK — Query the fixed pipeline with 3 economic questions
# -----------------------------------------------------------

questions = [
    # Question 1: Should require info from multiple meetings
    "How did the Fed's stance on inflation evolve throughout 2023, and how did the policy response change across meetings?",

    # Question 2: Should test specificity (a narrow, factual question)
    "What was the federal funds rate target range in September 2023?",

    # Question 3: Should test reasoning across documents
    "What risks did FOMC participants identify as threats to the economic outlook, and did these risks change over the year?",
]

for i, q in enumerate(questions, 1):
    print(f'\n{"="*60}')
    print(f'Question {i}: {q}')
    print(f'{"="*60}')

    result = rag_answer(q, retriever_fixed)

    print(f'\nAnswer: {result["result"]}')
    print(f'\nSources ({len(result["source_documents"])}):')
    for j, doc in enumerate(result["source_documents"]):
        print(f'  [{j+1}] {doc.metadata.get("meeting", "unknown")} — '
              f'"{doc.page_content[:80]}..."')

    # YOUR EVALUATION (fill in after running):
    print(f'\n--- Your Evaluation ---')
    print(f'Retrieval quality (1-5): Answerd below in the Markdown')
    print(f'Answer grounding (1-5): Answerd below in the Markdown')
    print(f'Completeness (1-5):Answerd below in the Markdown')
    print(f'Notes: Answerd below in the Markdown ')


Question 1: How did the Fed's stance on inflation evolve throughout 2023, and how did the policy response change across meetings?

Answer: Based on 4 retrieved FOMC minute(s) (January 2023, September 2023, June 2023, December 2023), the relevant context is:

- [January 2023] Federal Open Market Committee - January 2023 Meeting.
        Participants noted that inflation remained well above the Committee's
        2 percent objective. The labor market remained very tight, w...
- [September 2023] Federal Open Market Committee - September 2023 Meeting.
        Participants discussed the uncertainty surrounding the economic outlook
        and noted various risks. Supply-side improvements had co...
- [June 2023] Federal Open Market Committee - June 2023 Meeting.
        Participants noted that the pace of job gains had slowed but remained
        solid. Consumer spending appeared to have picked up. The housin...
- [December 2023] Federal Open Market Committee - December 2023 Meeting.
     

**Q1 Evaluation (Retrieval quality 5, Answer Grounding 5, Completeness 4):**


1.   It rerieved perfectly to the chunk of fourth meetings in 2023 of Januanry, June, September, and December. Each cite come directly from the according original paragraphs from the meeting, no made-up information.
2.   The timeline is complete from Inflation is 2% higher in January and increase interest rate to to December to the decrease in interest rate in 2024 with anti-inflation process maintained.
3. Information for four meetings are complete, stance for anti-inflation and policy are covered. However, rag_answer only checked first 200 characters, which some of the details got cut off.


**Q2 Evaluation (Retrieval quality 5, Answer Grounding 5, Completeness 4):**


1.   Source 1 retrieved accurately on the maintain the target range...chunk.
2.   While k=4 seems extra, and source 2 and 3 are not realy related, source 1 hit the target directly. Completeness
3.  4 out of 5 due to the 200 characters cutoff for the 5-1/4 to 5-1/2 percent, which not shown even it is in the original chunk.



**Q3 Evaluation (Retrieval quality 5, Answer Grounding 4, Completeness 3):**

1.   Risks of excessive tightening-January, Bank industry pressure-March, weak housing market-June, and future uncertainty-Sept are shown well by the retrieve.
2.   4/5 since rag_answer only listed the retrieved result and did not conduct comprehensive analysis of risk change over time.
3.   3/5 since no conclusion while with enough elements. The reason is due to the free version's restriction, however, since pre-lab requirement instead of ChatOpenAi API.

---

## EXTEND: RAG vs. Direct LLM Comparison

For each of your 3 questions, also ask the LLM **without** retrieval (no source documents). Compare the answers to assess whether RAG grounding improves accuracy and reduces hallucination.

In [8]:
# -----------------------------------------------------------
# YOUR TASK — Compare RAG vs. direct LLM answers
# -----------------------------------------------------------

# Simulated direct LLM (no retrieval, generic training-data style answer)
# Since we don't have an OpenAI API key (per Pre-Lab Preparation using
# sentence-transformers), we simulate the direct-LLM baseline with a
# generic, ungrounded response to illustrate the conceptual contrast.
def simulated_direct_llm(query):
    return (
        "The Federal Reserve raised interest rates several times in 2023 to combat high inflation, "
        "which had reached multi-decade highs. The Fed eventually paused rate hikes later in the year "
        "as inflation began to come down. Risks discussed by the FOMC typically include inflation, "
        "labor market conditions, and geopolitical tensions. "
        "[Note: this answer reflects general knowledge; it does NOT cite any specific FOMC meeting.]"
    )

for i, q in enumerate(questions, 1):
    print(f'\n{"="*60}')
    print(f'Question {i}: {q}')
    print(f'{"="*60}')

    # RAG answer (from fixed pipeline)
    rag_result = rag_answer(q, retriever_fixed)
    rag_answer_text = rag_result["result"]

    # Direct LLM answer (no retrieval)
    direct_answer = simulated_direct_llm(q)

    print(f'\nRAG Answer:')
    print(f'  {rag_answer_text[:300]}...' if len(rag_answer_text) > 300 else f'  {rag_answer_text}')

    print(f'\nDirect LLM Answer:')
    print(f'  {direct_answer[:300]}...' if len(direct_answer) > 300 else f'  {direct_answer}')


Question 1: How did the Fed's stance on inflation evolve throughout 2023, and how did the policy response change across meetings?

RAG Answer:
  Based on 4 retrieved FOMC minute(s) (January 2023, September 2023, June 2023, December 2023), the relevant context is:

- [January 2023] Federal Open Market Committee - January 2023 Meeting.
        Participants noted that inflation remained well above the Committee's
        2 percent objective. Th...

Direct LLM Answer:
  The Federal Reserve raised interest rates several times in 2023 to combat high inflation, which had reached multi-decade highs. The Fed eventually paused rate hikes later in the year as inflation began to come down. Risks discussed by the FOMC typically include inflation, labor market conditions, an...

Question 2: What was the federal funds rate target range in September 2023?

RAG Answer:
  Based on 4 retrieved FOMC minute(s) (September 2023, December 2023, January 2023, September 2023), the relevant context is:

- [Sept

Q1:
RAG cited each four meetings within 2023 (Jan-June-Sept-Dec). While direct LLM is general description like Fed increase interest rate then paused, which lack of detail and could be utilized in every contraction period. RAG is more specific and supported with sources, LLM has the accuracy and hallucination risk.


Q2:
RAG retrieved the chunk of "maintain the target range for the federal fund." Direct LLM did not list out source and did not issued any target range number. While both are on the right track, RAG is able to verified. Direct LLM has the risk of hallucination.


Q3:
RAG significantly more advanced. RAG listed out risks of excessive tightening-January, Bank industry pressure-March, weak housing market-June, and future uncertainty-Sept. However, LLM mentioned geopolitical reasons instead, which such words did not exist within the 2023 FOMC information, which is hallucination. Therefore, for this specific point, RAG significantly better.

---

## Final Reflection

Answer each question in 2-3 sentences.

### Q1: When does RAG outperform direct LLM prompting? When does it not matter?

*Your answer:*
RAG outperform direct LLM prompting when the question need details from the specific corpus, and when source attribution is required. For instance, policy memo and academic research. This conclusion came directly from the result of the Q3 from previous question, which LLM mentioned geopolitical reasons, words that not existed within the 2023 FOMC information. RAG is more stable since it is fixed within the document. RAG is not matter when the question is related into a broad concept, since LLM already trained and adding retrieve only added up the cost and time.


### Q2: How do chunking parameters affect retrieval quality?

*Connect to the bias-variance tradeoff from Ch 15 — small chunks have high variance (noisy retrieval), large chunks have high bias (diluted relevance).*

*Your answer:*
Small chunks means high variance, for instance chunk_size=50. 97 fragement, average 29 characters. In other words, it is too short to hold the whole phrases meaning, leading to a high noise of top-k results. This result came from the Error 1 cell in the previous question.
Large chunks means high bias. To be specific, a 2000 characters chunk that discussed inflation and employment at the same time would lead to both hit on the queries, which result in less correlation and prompt filled with unrelated information. Eventually, it is realted to the same issue within Ch15, the cross-validation condition of choosing model between noise retrieve and retrieval dilution.


### Q3: In what economic research contexts would you use RAG over direct prompting?

*Think about: policy analysis on proprietary documents, literature review, regulatory compliance, central bank communication analysis.*

*Your answer:*
Research within the enterprise. To be specific, internal report analysis and customers information are not stored in any LLM.


Supervising and compliance works. To be specific, based on Federal Register and Dodd-Frank rulemaking to conduct analysis. LLM generation is not one of the accepted citation.

Systematic literature review. To be specific, conducting research within the defined working papers, each conclusions have to be able to retrieve back to the specific article.

Central bank communication and analysis. To be specific, it would involve FOMC summary analysis, ECB statements, press conferences and so on.


---

## Submission

Submit the following on Canvas:

1. **Deployed portfolio URL** (Vercel link)
2. **This notebook** with all code cells run, evaluations filled in, and reflections answered
3. **GitHub repository link** for the portfolio source code

```bash
cd econ-portfolio
git add .
git commit -m "Lab 25: AI Literacy — Portfolio + RAG Pipeline"
git push origin main
```